# Model evaluation pipeline

This notebook runs the **decision-useful evaluation workflow** on holdout (test) predictions:

1. **Task detection** — regression vs classification
2. **Headline metrics** — MAE, RMSE, R² (regression)
3. **Core diagnostic plots** — predicted vs actual, residuals vs predicted, residual distribution
4. **Subgroup performance** — error by subgroup (e.g. state) when columns are available
5. **Time-based performance** — actual vs predicted over time when a time column exists
6. **Top error inspection** — largest absolute errors with key features
7. **Interpretation** — what the diagnostics suggest and recommended next steps

Uses **test-set predictions only** (no training evaluation).

## Setup and inputs

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Allow importing evaluation when kernel cwd is repo root
if (Path.cwd() / "modeling" / "evaluation.py").exists():
    sys.path.insert(0, str(Path.cwd() / "modeling"))
from evaluation import evaluate, detect_task, regression_subgroup_stats

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "modeling" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "modeling" / "results"

# Which model's predictions to evaluate (file under results/<TARGET>/)
TARGET = "CASTHMA"
MODEL_NAME = "xgboost"  # e.g. ridge, lasso, elasticnet, xgboost
PRED_PATH = RESULTS_DIR / TARGET / f"{MODEL_NAME}_predictions.csv"
TEST_PATH = DATA_DIR / f"test_{TARGET}.csv"

## Load predictions and align test data

In [ ]:
pred_df = pd.read_csv(PRED_PATH)
print("Predictions:", pred_df.shape)
print(pred_df.head())

y_true = pred_df["actual"].values
y_pred = pred_df["prediction"].values

# Optional: merge with test set for subgroups and time
X_test = None
subgroup_columns = {}
time_column = None

def _qsplit(series, *, q=0.5, high_label="high", low_label="low", invert=False):
    s = pd.to_numeric(series, errors="coerce")
    thr = s.quantile(q)
    if invert:
        # e.g. lower income => higher poverty
        out = np.where(s <= thr, high_label, low_label)
    else:
        out = np.where(s >= thr, high_label, low_label)
    out[pd.isna(s)] = None
    return out

if TEST_PATH.exists():
    test_df = pd.read_csv(TEST_PATH)
    # Align by (FIPS, YEAR)
    merge_cols = [c for c in ["FIPS", "YEAR"] if c in pred_df.columns and c in test_df.columns]
    if merge_cols:
        merged = pred_df.merge(test_df, on=merge_cols, how="left")
        X_test = merged

        # ---- EJ subgroup definitions (county-level covariates) ----
        def _string_labels(s, missing_label="missing"):
            out = pd.Series(s).astype("string").fillna(missing_label)
            return out.astype(str).values

        # Poverty: prefer explicit poverty rate; otherwise proxy with median_income (low income ~ high poverty).
        if "poverty_rate" in merged.columns:
            merged["ej_poverty"] = _qsplit(merged["poverty_rate"], high_label="high_poverty", low_label="low_poverty")
            subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])
        elif "pct_poverty" in merged.columns:
            merged["ej_poverty"] = _qsplit(merged["pct_poverty"], high_label="high_poverty", low_label="low_poverty")
            subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])
        elif "median_income" in merged.columns:
            merged["ej_poverty"] = _qsplit(merged["median_income"], high_label="high_poverty", low_label="low_poverty", invert=True)
            subgroup_columns["ej_poverty"] = _string_labels(merged["ej_poverty"])

        # SVI (if present)
        for svi_col in ["SVI", "svi", "svi_total", "svi_overall", "svi_rank"]:
            if svi_col in merged.columns:
                merged["ej_svi"] = _qsplit(merged[svi_col], high_label="high_SVI", low_label="low_SVI")
                subgroup_columns["ej_svi"] = _string_labels(merged["ej_svi"])
                break

        # Majority nonwhite: proxy via pct_white if available (pct_white < 50).
        if "pct_white" in merged.columns:
            pw = pd.to_numeric(merged["pct_white"], errors="coerce")
            merged["ej_majority_nonwhite"] = np.where(pw < 50, "majority_nonwhite", "not_majority_nonwhite")
            subgroup_columns["ej_majority_nonwhite"] = _string_labels(merged["ej_majority_nonwhite"])

        # Rural vs urban: use NCHS urban-rural (1=most urban ... 6=most rural).
        if "nchs_urban_rural" in merged.columns:
            ur = pd.to_numeric(merged["nchs_urban_rural"], errors="coerce")
            merged["ej_rural"] = np.where(ur >= 5, "rural", "urban")
            subgroup_columns["ej_rural"] = _string_labels(merged["ej_rural"])

        # High vs low pesticide use: total kg per cropland acre when possible.
        if "pesticide_total_kg" in merged.columns:
            denom = pd.to_numeric(merged.get("cropland_acres"), errors="coerce")
            num = pd.to_numeric(merged["pesticide_total_kg"], errors="coerce")
            per_acre = num / denom.replace(0, np.nan)
            merged["ej_pesticide_use"] = _qsplit(per_acre, high_label="high_pesticide_use", low_label="low_pesticide_use")
            subgroup_columns["ej_pesticide_use"] = _string_labels(merged["ej_pesticide_use"])

        # High vs low IPM breadth: prefer per-acre breadth if present.
        if "ipm_breadth_acre" in merged.columns:
            merged["ej_ipm_breadth"] = _qsplit(merged["ipm_breadth_acre"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
            subgroup_columns["ej_ipm_breadth"] = _string_labels(merged["ej_ipm_breadth"])
        elif "ipm_breadth_value" in merged.columns:
            merged["ej_ipm_breadth"] = _qsplit(merged["ipm_breadth_value"], high_label="high_ipm_breadth", low_label="low_ipm_breadth")
            subgroup_columns["ej_ipm_breadth"] = _string_labels(merged["ej_ipm_breadth"])

        # Keep state_fips subgroup too (useful context)
        if "state_fips" in merged.columns:
            subgroup_columns["state_fips"] = merged["state_fips"].values

        if "YEAR" in merged.columns:
            time_column = "YEAR"
    print("Merged test rows:", len(merged) if merge_cols else 0)

task = detect_task(y_true)
print("Task detected:", task)

## Run evaluation

In [ ]:
result = evaluate(
    y_true,
    y_pred,
    task=task,
    X_test=X_test,
    subgroup_columns=subgroup_columns if subgroup_columns else None,
    time_column=time_column,
    top_n_errors=20,
    min_subgroup_support=5,
)

print("Task:", result["task_detected"])

## 1. Headline metrics

In [ ]:
display(result["metrics"])

## 2. Core diagnostic plots

## 4b. Map: prediction error by place (county)

Choropleth of **mean absolute error** by county; optionally highlight counties with **most errors** vs **best predictions**.

In [ ]:
# Build per-county error stats from predictions (works even without X_test)
pred_df["abs_residual"] = np.abs(pred_df["actual"] - pred_df["prediction"])
fips5 = lambda s: s.astype(str).str.zfill(5)
county_error = (
    pred_df.groupby(pred_df["FIPS"].astype(str).str.zfill(5))
    .agg(mean_abs_error=("abs_residual", "mean"), n=("abs_residual", "count"))
    .reset_index()
)
county_error["FIPS"] = fips5(county_error["FIPS"].astype(str))

# Optional: label worst vs best (e.g. top/bottom 20% of counties by mean error)
q_high = county_error["mean_abs_error"].quantile(0.80)
q_low = county_error["mean_abs_error"].quantile(0.20)
county_error["error_tier"] = "middle"
county_error.loc[county_error["mean_abs_error"] >= q_high, "error_tier"] = "worst"
county_error.loc[county_error["mean_abs_error"] <= q_low, "error_tier"] = "best"

# Load US counties GeoJSON for choropleth
import urllib.request
import json
import plotly.express as px

COUNTIES_GEOJSON_URL = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
with urllib.request.urlopen(COUNTIES_GEOJSON_URL) as url:
    counties_geojson = json.loads(url.read())

# Map 1: continuous color by mean absolute error (red = high error, green = low)
fig1 = px.choropleth(
    county_error,
    geojson=counties_geojson,
    locations="FIPS",
    color="mean_abs_error",
    scope="usa",
    featureidkey="id",
    color_continuous_scale="RdYlGn_r",  # red = high error, green = low
    title=f"Mean absolute prediction error by county ({TARGET})",
)
fig1.update_geos(fitbounds="locations", visible=False)
fig1.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=450)
fig1.show()

# Map 2: categorical — worst (red), best (green), middle (gray)
fig2 = px.choropleth(
    county_error,
    geojson=counties_geojson,
    locations="FIPS",
    color="error_tier",
    scope="usa",
    featureidkey="id",
    category_orders={"error_tier": ["worst", "middle", "best"]},
    color_discrete_map={"worst": "#e74c3c", "middle": "#bdc3c7", "best": "#2ecc71"},
    title=f"Worst / best prediction counties (top & bottom 20% by mean error) — {TARGET}",
)
fig2.update_geos(fitbounds="locations", visible=False)
fig2.update_layout(margin=dict(l=0, r=0, t=40, b=0), height=450)
fig2.show()

In [ ]:
for fig, title in result["figures"]:
    fig.suptitle(title, y=1.02)
    plt.show()

## 3. Subgroup performance (if available)

In [ ]:
for name, tbl in result["subgroup_tables"].items():
    print(f"Subgroup: {name}")
    display(tbl)
    print()

## D. EJ subgroup analysis

County-level subgroup tables stratified by EJ-relevant dimensions.

**Residual = actual − predicted** (positive => underprediction; negative => overprediction).


In [ ]:
# Build a county-level table (one row per FIPS)
if X_test is None or "FIPS" not in X_test.columns:
    raise ValueError("X_test with FIPS is required for EJ subgroup analysis. Make sure TEST_PATH merges.")

county_pred = (
    pred_df.assign(FIPS=pred_df["FIPS"].astype(str).str.zfill(5))
    .groupby("FIPS")
    .agg(mean_actual=("actual", "mean"), mean_predicted=("prediction", "mean"))
    .reset_index()
)
county_pred["residual"] = county_pred["mean_actual"] - county_pred["mean_predicted"]
county_pred["abs_residual"] = county_pred["residual"].abs()

# Pull (roughly) county-constant covariates from X_test
X_test2 = X_test.copy()
X_test2["FIPS"] = X_test2["FIPS"].astype(str).str.zfill(5)
county_covars = X_test2.sort_values(["FIPS", "YEAR"] if "YEAR" in X_test2.columns else ["FIPS"]).drop_duplicates("FIPS")

county = county_pred.merge(county_covars, on="FIPS", how="left")

# Use population weighting when available (public-health relevant)
weights = county["population"].values if "population" in county.columns else None

ej_group_cols = [c for c in ["ej_poverty", "ej_svi", "ej_majority_nonwhite", "ej_rural", "ej_pesticide_use", "ej_ipm_breadth"] if c in county.columns]
if not ej_group_cols:
    raise ValueError("No EJ subgroup columns found. Check column names in test data (poverty/SVI/rurality/race/IPM/pesticides).")

for col in ej_group_cols:
    tbl = regression_subgroup_stats(
        county["mean_actual"].values,
        county["mean_predicted"].values,
        groups=county[col].values,
        sample_weight=weights,
    ).sort_values("mean_residual", ascending=False)
    print(f"EJ subgroup: {col}")
    display(tbl)
    print()


## E. High-burden county profiling

Compare the **top decile** vs **bottom decile** of counties by *predicted* asthma burden.


In [ ]:
county_rank = county.copy()
county_rank = county_rank.dropna(subset=["mean_predicted"])
q90 = county_rank["mean_predicted"].quantile(0.90)
q10 = county_rank["mean_predicted"].quantile(0.10)
county_rank["burden_group"] = np.where(county_rank["mean_predicted"] >= q90, "top_decile", np.where(county_rank["mean_predicted"] <= q10, "bottom_decile", None))
profile = county_rank[county_rank["burden_group"].notna()].copy()

# Candidate profile columns (only keep those that exist)
candidate_cols = [
    # IPM / documentation
    "ipm_breadth_acre", "ipm_breadth_value", "chemical_reliance_acre", "chemical_reliance_value",
    "ipm_doc_coverage_share", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age",
    "ipm_primary_match_tier",
    # Pesticides (macro)
    "pesticide_total_kg", "pesticide_respiratory_kg",
    # Crop mix / ag
    "pct_cropland", "cropland_acres", "cropland_diversity", "specialty_crop_share", "county_crop_concentration", "total_ag_value",
    "corn_acres", "soybean_acres", "cotton_acres", "wheat_acres", "hay_acres", "fruit_veg_acres", "rice_acres", "sorghum_acres",
    # EJ / demographics
    "median_income", "median_age", "population", "pct_white", "pct_black", "pct_asian", "pct_hispanic", "nchs_urban_rural",
    # Health context
    "CSMOKING", "OBESITY", "DIABETES",
]
profile_cols = [c for c in candidate_cols if c in profile.columns]

prof_tbl = profile.groupby("burden_group")[profile_cols].mean(numeric_only=True).T
if "top_decile" in prof_tbl.columns and "bottom_decile" in prof_tbl.columns:
    prof_tbl["top_minus_bottom"] = prof_tbl["top_decile"] - prof_tbl["bottom_decile"]
display(prof_tbl.sort_values("top_minus_bottom", ascending=False, na_position="last"))


## F. Error analysis through an EJ lens

Ask whether large errors align with social vulnerability or with data/documentation coverage.


In [ ]:
# County-level error (abs_residual) vs documentation / crop specialization / pesticides
err = county.copy()
err = err.dropna(subset=["abs_residual"])

def _safe_corr(x, y):
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    m = ~(pd.isna(x) | pd.isna(y))
    if m.sum() < 20:
        return np.nan
    return float(np.corrcoef(x[m], y[m])[0,1])

lens_vars = [
    "ipm_doc_coverage_share", "mean_text_quality", "mean_geo_confidence", "weighted_doc_age",
    "specialty_crop_share", "county_crop_concentration",
    "pesticide_total_kg", "pesticide_respiratory_kg",
    "ipm_breadth_acre", "chemical_reliance_acre",
    "median_income", "pct_white", "nchs_urban_rural",
]
lens_vars = [v for v in lens_vars if v in err.columns]

corrs = pd.DataFrame({
    "variable": lens_vars,
    "corr_with_abs_error": [ _safe_corr(err[v], err["abs_residual"]) for v in lens_vars ],
}).sort_values("corr_with_abs_error", ascending=False)
display(corrs)

# Top error counties with EJ-relevant fields
show_cols = [c for c in [
    "FIPS", "NAME", "state_fips", "mean_actual", "mean_predicted", "residual", "abs_residual",
    "ej_poverty", "ej_svi", "ej_majority_nonwhite", "ej_rural", "ej_pesticide_use", "ej_ipm_breadth",
    "ipm_doc_coverage_share", "mean_text_quality", "mean_geo_confidence", "specialty_crop_share",
] if c in err.columns]
display(err.sort_values("abs_residual", ascending=False).head(25)[show_cols])


## 4. Top error inspection

In [ ]:
display(result["top_errors"])

## 5. Interpretation and next steps

In [ ]:
print("Interpretation:")
print(result["interpretation"])
print()
print("Recommended next steps:")
print(result["next_steps"])

## Optional: save figures

In [ ]:
SAVE_FIGURES = False  # set True to write PNGs
if SAVE_FIGURES:
    out_dir = RESULTS_DIR / TARGET / "evaluation"
    out_dir.mkdir(parents=True, exist_ok=True)
    for fig, title in result["figures"]:
        fig.savefig(out_dir / f"{title}.png", dpi=150, bbox_inches="tight")
    print("Saved to", out_dir)